# 05 — RAG Briefing Generator

**Maps from previous project:** Notebook 05 (corpus → FAISS → LLM briefing)

**Input:** companies, opportunities, recommendations, sector guides

**Tasks:**
1. Build a document corpus (company profiles, opportunities, sector guides)
2. Embed into FAISS
3. Generate personalised leavers’ briefings (OpenAI if key present, else offline template)

**Output:** `app/app_data/faiss_index/`, sample briefings in `data/processed/sample_briefings.md`

In [2]:
import os
import pickle
import sys
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from glos_recommender.briefing import build_briefing_prompt, fallback_briefing
from glos_recommender.matching import build_leaver_profile

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CORPUS_DIR = PROJECT_ROOT / "data" / "corpus"
FAISS_DIR = PROJECT_ROOT / "app" / "app_data" / "faiss_index"
MODEL_DIR = PROJECT_ROOT / "models" / "local_minilm_model"

CORPUS_DIR.mkdir(parents=True, exist_ok=True)
FAISS_DIR.mkdir(parents=True, exist_ok=True)
load_dotenv(PROJECT_ROOT / ".env")

companies = pd.read_pickle(PROCESSED_DIR / "company_feature_store.pkl")
opportunities = pd.read_pickle(PROCESSED_DIR / "opportunities_master.pkl")
recommendations = pd.read_csv(PROCESSED_DIR / "recommendations.csv")
leavers = pd.read_pickle(PROCESSED_DIR / "clustered_leavers.pkl")
print(f"Companies={len(companies)} opps={len(opportunities)} recs={len(recommendations)}")

Companies=1139 opps=2192 recs=24


## 1. Build document corpus

In [3]:
# Prefer scripts/04_export_rag_corpus.py for Source: provenance lines.
# Inline export kept here for notebook self-containment.
from glos_recommender.labels import clean_company_summary
from glos_recommender.provenance import classify_employer_source, source_label

company_docs = []
for _, r in companies.iterrows():
    data = r.to_dict()
    kind = classify_employer_source(data)
    summary = clean_company_summary(data.get("summary"))
    caveat = ""
    if kind == "vacancies":
        caveat = " Vacancy data may be historical or closed."
    elif kind == "companies_house":
        caveat = " Registry facts only — not a careers page."
    company_docs.append(
        f"Company: {data.get('name')} ({data.get('town')}). "
        f"Sectors: {data.get('sectors')}. Entry routes: {data.get('entry_routes')}. "
        f"Roles: {data.get('role_families')}. {summary}{caveat} "
        f"Website: {data.get('website') or 'none on file'}. "
        f"Source: {source_label(kind)}."
    )

(CORPUS_DIR / "company_profiles.txt").write_text("\n\n".join(company_docs), encoding="utf-8")

opp_docs = []
for _, o in opportunities.iterrows():
    cname = companies.loc[companies["company_id"] == o["company_id"], "name"]
    cname = cname.iloc[0] if len(cname) else o["company_id"]
    src = str(o.get("source") or "").strip().lower()
    if src == "vacancies":
        source_line = "Find an apprenticeship open data (may be historical / closed)"
    elif src in {"seed", "curated", "manual"}:
        source_line = "Curated opportunity profile"
    else:
        source_line = source_label(src or "seed")
    opp_docs.append(
        f"Opportunity at {cname}: {o['title']} ({o['entry_route']}, {o['level']}). "
        f"{o['description']} Typical quals: {o['typical_quals']}. "
        f"Useful projects: {o['useful_projects']}. Tips: {o['application_tips']}. "
        f"Source: {source_line}."
    )

(CORPUS_DIR / "opportunities.txt").write_text("\n\n".join(opp_docs), encoding="utf-8")

print(f"Company docs: {len(company_docs)}")
print(f"Opportunity docs: {len(opp_docs)}")
print(f"Sector guides present: {(CORPUS_DIR / 'sector_guides.txt').exists()}")


Company docs: 1139
Opportunity docs: 2192
Sector guides present: True


## 2. Chunk, embed, FAISS index

In [4]:
def chunk_text(text, chunk_size=180, overlap=30):
    words = text.split()
    chunks, start = [], 0
    while start < len(words):
        chunks.append(" ".join(words[start:start + chunk_size]))
        start += max(1, chunk_size - overlap)
    return chunks

all_chunks, sources = [], []
for fpath in CORPUS_DIR.glob("*.txt"):
    for para in [p.strip() for p in fpath.read_text(encoding="utf-8").split("\n\n") if p.strip()]:
        for chunk in chunk_text(para):
            all_chunks.append(chunk)
            sources.append(fpath.stem)

print(f"Chunks: {len(all_chunks)}")
print(pd.Series(sources).value_counts().to_dict())

Chunks: 3335
{'opportunities': 2192, 'company_profiles': 1139, 'sector_guides': 4}


In [5]:
if (MODEL_DIR / "config.json").exists():
    embedder = SentenceTransformer(str(MODEL_DIR))
else:
    embedder = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedder.encode(all_chunks, show_progress_bar=True, batch_size=64)
embeddings = np.asarray(embeddings, dtype="float32")
faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

faiss.write_index(index, str(FAISS_DIR / "corpus.index"))
with open(FAISS_DIR / "chunks.pkl", "wb") as f:
    pickle.dump({"chunks": all_chunks, "sources": sources}, f)

print(f"FAISS index: {index.ntotal} vectors → {FAISS_DIR}")

Batches: 100%|██████████| 53/53 [00:54<00:00,  1.02s/it]

FAISS index: 3335 vectors → c:\Users\MSI Katana Gaming\HigherED_ML_app\app\app_data\faiss_index


## 3. Retrieve + generate briefings for sample matches

In [6]:
def retrieve(query, top_k=4):
    q = embedder.encode([query])
    q = np.asarray(q, dtype="float32")
    faiss.normalize_L2(q)
    scores, idxs = index.search(q, top_k)
    return [
        {"chunk": all_chunks[i], "source": sources[i], "score": float(scores[0][j])}
        for j, i in enumerate(idxs[0])
    ]

api_key = os.getenv("OPENAI_API_KEY")
client = None
if api_key:
    from openai import OpenAI
    client = OpenAI(api_key=api_key)
    print("OpenAI client ready")
else:
    print("No OPENAI_API_KEY — using offline fallback briefings")

OpenAI client ready


In [7]:
sample_recs = recommendations[recommendations["rank"] == 1].head(3)
briefing_parts = []

for _, rec in sample_recs.iterrows():
    leaver_row = leavers[leavers["leaver_id"] == rec["leaver_id"]].iloc[0]
    company_row = companies[companies["company_id"] == rec["company_id"]].iloc[0]
    profile = build_leaver_profile(leaver_row["form"])

    query = (
        f"{company_row['name']} {company_row['sectors']} opportunities for "
        f"{leaver_row['leaver_type']} interested in {leaver_row['interest_sectors']}"
    )
    context = retrieve(query, top_k=4)

    if client:
        system, user = build_briefing_prompt(profile, company_row, opportunities, context)
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
            temperature=0.4,
            max_tokens=900,
        )
        text = resp.choices[0].message.content
    else:
        text = fallback_briefing(profile, company_row, opportunities)

    block = f"# {rec['leaver_id']} → {rec['company_name']}\n\n{text}\n\n---\n"
    briefing_parts.append(block)
    print(block[:500], "...\n")

out_md = PROCESSED_DIR / "sample_briefings.md"
out_md.write_text("\n".join(briefing_parts), encoding="utf-8")
print(f"Saved {out_md}")

# L001 → AtkinsRéalis

### Why this company fits you
AtkinsRéalis is a fantastic match for you because it operates in the cyber and digital sector, which aligns perfectly with your interests in cybersecurity, digital defence, and software development. As a local employer in Cheltenham, they understand the unique dynamics of the Gloucestershire tech scene, which is an exciting hub for cyber innovation. Your investigative and conventional work style is well-suited to the roles they offer, allowing ...

# L002 → CANAL & RIVER TRUST

### Why this company fits you
Canal & River Trust is a fantastic match for you because it operates in sectors that align perfectly with your interests in aerospace, advanced manufacturing, and mechanical/electrical engineering. Their focus on maintaining and enhancing waterways means you’ll be working on real-world engineering challenges, which is something you’re passionate about. The company values practical skills and teamwork, which fits your realistic and

## 4. Pipeline complete

In [8]:
print("=" * 50)
print("RAG PIPELINE COMPLETE")
print("=" * 50)
print("Launch the app with:")
print("  streamlit run app/app.py")

RAG PIPELINE COMPLETE
Launch the app with:
  streamlit run app/app.py
